In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

from astropy.table import Table, join, QTable, vstack
import astropy.units as u
import sys
import pyneb as pn
from multiprocessing import Pool
import multiprocessing as mp
import math
from astropy.io import fits
from orcs.process import SpectralCube

from astropy.nddata import NDData, Cutout2D
from astropy.wcs import WCS

import astropy
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.wcs.utils import pixel_to_skycoord, skycoord_to_pixel

from reproject import reproject_interp
import reproject
from regions import PixCoord

import pylab as pl

from scipy.optimize import curve_fit
from scipy.integrate import quad
from scipy.interpolate import griddata

from orb.fit import fit_lines_in_spectrum
from orb.utils.spectrum import corr2theta, amp_ratio_from_flux_ratio
from orb.core import Lines
import gvar
import orb

import extinction
from extinction import apply, remove

from photutils.detection import DAOStarFinder

import aplpy
import seaborn as sns

from skimage import filters

import statsmodels.api as sm
from scipy.stats import spearmanr

import sys
sys.path.append("/home/habjan/SITELLE/sitelle_metallicities")
import analysis_functions as af

### Font style

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",  # <- key change
    "text.usetex": False,               # just to be sure
})

### Figure 1: auroral line fits

In [ ]:
galaxynum = 4
galdic = {1:'NGC4254', 2:'NGC4535', 3:'NGC3351', 4:'NGC2835', 5:'NGC0628', 6:'NGC3627'}  #There is no SITELLE data for NGC 4254, NGC 2835 has the best data 
galaxy = galdic[galaxynum]

galveldic = {'NGC4254': 2388 , 'NGC4535': 1954  , 'NGC3351': 775, 'NGC2835': 867, 'NGC0628':651, 'NGC3627':715}
galvel = galveldic[galaxy]

hdul = fits.open(f'/home/habjan/SITELLE/sandbox_notebooks/pdf_fit_plotting/{galaxy}_spectra_fitting.fits')

fit5755 = hdul[6].data
snr5755 = hdul[7].data

fit6312 = hdul[8].data
snr6312 = hdul[9].data

fit7319 = hdul[10].data
snr7319 = hdul[11].data

fit7330 = hdul[12].data
snr7330 = hdul[13].data

hdu1 = fits.open(f'/home/habjan/SITELLE/data/data_raw_intermediate/{galaxy}_VorSpectra.fits')
hdu2 = fits.open(f'/home/habjan/SITELLE/data/data_raw_intermediate/{galaxy}_ppxf-bestfit-emlines.fits')

bestfit_gas = np.ma.MaskedArray( hdu2[1].data["BESTFIT"],
                                mask=hdu2[1].data['BESTFIT']==0)

gas_templ = np.ma.MaskedArray( hdu2[1].data["GAS_BESTFIT"],
                              mask=hdu2[1].data['BESTFIT']==0)

lam = np.exp(hdu1[2].data['LOGLAM'])
log_spec = hdu1[1].data['SPEC']

fluxin = log_spec - (bestfit_gas - gas_templ)

def gaussian(x, a, w0, sigma):    #a:amplitude, wavelength:feature wavelength, sigma:spectral resolution, C:zero offeset
        return a * np.exp((-(x-w0) ** 2)/ (2 * sigma ** 2)) + C + lp*x

In [ ]:
i = 144

wave_surr = 50
flux_bottom = -0.5
flux_top = 1.3

fig, (ax1, ax2, ax3)  = plt.subplots(1, 3)
fig.set_figheight(5)
fig.set_figwidth(14)

### [NII]5755
ax1.plot(lam, fluxin[i], c='k')
lp, C = fit5755[i][4], fit5755[i][3]
ax1.plot(lam, gaussian(lam, fit5755[i][0], fit5755[i][1], fit5755[i][2]), c='blue')
ax1.axvline(fit5755[i][1], linestyle='--', c='blue', alpha=0.5, label=r'[NII] $\lambda$5755')
ax1.set_xlim(fit5755[i][1] - wave_surr , fit5755[i][1] + wave_surr)
ax1.set_ylim(flux_bottom* fit5755[i][0]*0.5, flux_top*fit5755[i][0])
ax1.fill_between(lam[np.where(lam > fit5755[i][1] - 175)[0][0]:np.where(lam > fit5755[i][1] - 8)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax1.fill_between(lam[np.where(lam > fit5755[i][1] + 180)[0][0]:np.where(lam > fit5755[i][1] + 400)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax1.fill_between(lam[np.where(lam > fit5755[i][1] + 8)[0][0]:np.where(lam > fit5755[i][1] + 100)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax1.legend(loc='upper right', fontsize=12)
ax1.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

### [SIII]6312
ax2.plot(lam, fluxin[i], c='k')
lp, C = fit6312[i][4], fit6312[i][3]
ax2.plot(lam, gaussian(lam, fit6312[i][0], fit6312[i][1], fit6312[i][2]), c='green')
ax2.axvline(fit6312[i][1], linestyle='--', c='green', alpha=0.5, label=r'[SIII] $\lambda$6312')
ax2.set_xlim(fit6312[i][1] - wave_surr , fit6312[i][1] + wave_surr)
ax2.set_ylim(flux_bottom* fit6312[i][0], flux_top*fit6312[i][0])
ax2.fill_between(lam[np.where(lam > fit6312[i][1] - 200)[0][0]:np.where(lam > fit6312[i][1] - 70)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax2.fill_between(lam[np.where(lam > fit6312[i][1] + 63)[0][0]:np.where(lam > fit6312[i][1] + 120)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax2.fill_between(lam[np.where(lam > fit6312[i][1] + 17)[0][0]:np.where(lam > fit6312[i][1] + 45)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax2.legend(loc='upper right', fontsize=12)
ax2.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

### [OII]7320, 7330
ax3.plot(lam, fluxin[i], c='k')
lp, C = fit7319[i][4], fit7319[i][3]
ax3.plot(lam, gaussian(lam, fit7319[i][0], fit7319[i][1], fit7319[i][2]), c='darkorange')#, label='[OII]7320 Fit')
ax3.axvline(fit7319[i][1], linestyle='--', c='darkorange', alpha=0.5, label=r'[OII] $\lambda$7320')
ax3.set_xlim(np.mean([fit7319[i][1], fit7330[i][1]]) - wave_surr , np.mean([fit7319[i][1], fit7330[i][1]]) + wave_surr)
ax3.set_ylim(flux_bottom* fit7330[i][0], flux_top*fit7330[i][0])
ax3.fill_between(lam[np.where(lam > fit7319[i][1] - 150)[0][0]:np.where(lam > fit7319[i][1] - 10)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax3.fill_between(lam[np.where(lam > fit7319[i][1] + 20)[0][0]:np.where(lam > fit7319[i][1] + 150)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
lp, C = fit7330[i][4], fit7330[i][3]
ax3.plot(lam, gaussian(lam, fit7330[i][0], fit7330[i][1], fit7330[i][2]), c='red')#, label='[OII]7330 Fit')
ax3.axvline(fit7330[i][1], linestyle='--', c='red', alpha=0.5, label=r'[OII] $\lambda$7330')

ax3.legend(loc='upper right', fontsize=12)
ax3.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

fig.text(0.5, -0.02, r'Wavelength [$\AA$]', ha='center', fontsize=15)
fig.text(-0.02, 0.5, r'Flux [$\frac{erg}{cm^{2} \cdot s \cdot \AA \cdot 10^{-20}}$]', va='center', rotation='vertical', fontsize=15)
plt.tight_layout()
plt.show()

fig.savefig("/home/habjan/SITELLE/sitelle_metallicities/figures/auroral_lines.png", bbox_inches="tight")

# Physical Quantity plots

### Plotting styles

In [ ]:
plt.style.use('seaborn')

### Import strong line table

In [ ]:
products_data_path = '/home/habjan/SITELLE/data/data_products'

infile = open(products_data_path + f'/strong_line_MUSE+SITELLE.fits','rb')
strong_data = Table.read(infile)

galdic = {1:'NGC4254', 6:'NGC3627', 5:'NGC4535', 4:'NGC3351', 3:'NGC2835', 2:'NGC0628'}

### Function to find Spearman rank correlation coefficient and MC uncertainty

In [ ]:
def spear_func(in_x, in_y, in_x_err, in_y_err):
    spear = spearmanr(in_x, in_y)[0]
    spear_err = []
    for mc in range(10**3):
        try:
            spear_err.append(spearmanr(np.random.normal(in_x, in_x_err), np.random.normal(in_y, in_y_err))[0])
        except:
            continue
    return spear, np.nanstd(spear_err)

### Function to find WLS and MC uncertainty

In [ ]:
def WLS_func(in_x, in_y, in_x_err, in_y_err):

    weights = 1.0 / in_y_err**2
    X = sm.add_constant(in_x)
    model = sm.WLS(in_y, X, weights=weights)
    results = model.fit()
    WLS_coef, WLS_slope = results.params[0], results.params[1]
    mc_vals = np.array([sm.WLS(np.random.normal(in_y, in_y_err), sm.add_constant(np.random.normal(in_x, in_x_err)), weights=weights).fit().params for mc in range(10**3)])

    return WLS_coef, WLS_slope, np.nanstd(mc_vals[:, 0]), np.nanstd(mc_vals[:, 1])

### Function for MC uncertainty

In [ ]:
def quick_mc_err(quantity, quantity_err, iters):
    err_arr = np.array([np.random.normal(quantity, quantity_err) for mc in range(iters)])
    return np.nanstd(err_arr)

### Te-Te relationships

In [ ]:
def rv23_nii_siii(in_temp):
    temp = in_temp / 10**4
    out_temp = 1.35 * temp - 0.24
    return out_temp

def rv23_oii_siii(in_temp):
    temp = in_temp / 10**4
    out_temp = 0.67 * temp - 0.022
    return out_temp

def rv23_oii_nii(in_temp):
    temp = in_temp / 10**4
    out_temp = 0.47 * temp + 0.36
    return out_temp

def delgado_oii_nii(in_temp):
    return 0.62 * in_temp + 2660

def chaos_nii_siii(in_temp):
    temp = in_temp / 10**4
    out_temp = 1.46 * temp - 0.41
    return out_temp

def z21_nii_siii(in_temp):
    return 1.4 * in_temp - 3700

def z21_oii_siii(in_temp):
    return 0.67 * in_temp + 2200

def z21_oii_nii(in_temp):
    return 0.46 * in_temp + 4400

def b24_nii_siii(in_temp):
    return 1.22 * (in_temp) - 0.20

### Figure 2: Te-Te plot

In [ ]:
fig = plt.figure(figsize=(14, 14))

pq_list = ['OII_TEMP', 'NII_TEMP', 'SIII_TEMP']#, 'OIII_TEMP_MD23', 'OIII_TEMP_B12']#, 'SII_DEN_NII', 'SII_DEN_SIII']
pq_labels = [r'$T_{e, [O II]}$ [$\times 10^{4}$ K]', r'$T_{e, [N II]}$ [$\times 10^{4}$ K]', r'$T_{e, [S III]}$ [$\times 10^{4}$ K]'] 

colordic = {2: 'red', 3:'k', 4:'blue', 5:'green', 6:'m'}
galdic = {1:'NGC4254', 5:'NGC4535', 4:'NGC3351', 3:'NGC2835', 2:'NGC0628', 6:'NGC3627'}

snr = 0.9
axes = []
mc_count = 0
label_size = 18

snr = 3
mark_s = 35

s_line = np.linspace(-10, 10**5, 100)

fit_plot = []

for i in range(len(pq_list)):
    lim_data = []
    for j in range(len(pq_list)):
        if j - i < 0:
            if len(axes) == 0:
                axes.append(plt.subplot2grid((len(pq_list), len(pq_list)), (i, j)))
            else:
                axes.append(plt.subplot2grid((len(pq_list), len(pq_list)), (i, j) , sharex=axes[j], sharey=axes[j]))

            if i == len(pq_list) - 1:
                axes[-1].set_xlabel(pq_labels[j], fontsize=label_size)
            if j == 0:
                axes[-1].set_ylabel(pq_labels[i], fontsize=label_size)

            if j != 0:
                axes[-1].tick_params('y', labelleft=False)
            if i != len(pq_list) - 1:
                axes[-1].tick_params('x', labelbottom=False)

            plot_data_x, plot_data_x_err = [], []
            plot_data_y, plot_data_y_err = [], []

            for k in range(2, len(galdic)+1):

                plot_ind = np.where((strong_data['gal_name'] == galdic[k]) 
                        & (np.array(strong_data[pq_list[i]] / strong_data[pq_list[i]+'_ERR']) > snr) 
                        & (np.array(strong_data[pq_list[j]] / strong_data[pq_list[j]+'_ERR']) > snr)
                        & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]

                lim_data.append(np.array(strong_data[pq_list[i]][plot_ind]))
                lim_data.append(np.array(strong_data[pq_list[j]][plot_ind]))

                axes[-1].errorbar(strong_data[pq_list[j]][plot_ind] / 10**4, strong_data[pq_list[i]][plot_ind] / 10**4,
                            xerr=strong_data[pq_list[j]+'_ERR'][plot_ind] / 10**4, yerr=strong_data[pq_list[i]+'_ERR'][plot_ind] / 10**4,
                            fmt='none', ecolor='0.6', elinewidth=1, capsize=2, alpha=0.7, zorder=1)

                axes[-1].scatter(strong_data[pq_list[j]][plot_ind] / 10**4, strong_data[pq_list[i]][plot_ind] / 10**4,
                                  s=mark_s, c=colordic[k], edgecolor='k', linewidth=0.4, zorder=2, marker='D')
                
                div_f = 10**4
                plot_data_x = np.concatenate([plot_data_x, np.array(strong_data[pq_list[j]][plot_ind] / div_f)])
                plot_data_x_err = np.concatenate([plot_data_x_err, np.array(strong_data[pq_list[j]+'_ERR'][plot_ind] / div_f)])
                plot_data_y = np.concatenate([plot_data_y, np.array(strong_data[pq_list[i]][plot_ind] / div_f)])
                plot_data_y_err = np.concatenate([plot_data_y_err, np.array(strong_data[pq_list[i]+'_ERR'][plot_ind] / div_f)])
            
            spear, spear_err = spear_func(plot_data_x, plot_data_y, plot_data_x_err, plot_data_y_err)

            fit_coef, fit_slope, fit_coef_err, fit_slope_err = WLS_func(plot_data_x, plot_data_y, plot_data_x_err, plot_data_y_err)

            axes[-1].plot(s_line, fit_coef + fit_slope*s_line, 
                          c='darkorange', linestyle='--')
            axes[-1].fill_between(s_line, 
                                  (fit_coef - fit_coef_err) + (fit_slope - fit_slope_err)*s_line, 
                                  (fit_coef + fit_coef_err) + (fit_slope + fit_slope_err)*s_line,
                                  color='gold', alpha=0.3)
            
            axes[-1].text(0.59, 1.502, 
                          r'$\rho$ = 'f'{np.round(spear, decimals=3)} 'r'$\pm$ 'f'{np.round(spear_err, decimals=3)}',
                          fontsize = 12)
            
            axes[-1].plot(np.linspace(-1, 10**11, 10), np.linspace(-1, 10**11, 10), c='k', alpha=1)
            
            #Te-Te relations from other papers
            
            if pq_list[i] == 'SIII_TEMP' and pq_list[j] == 'NII_TEMP':
                
                plt.plot(s_line / 10**4, rv23_nii_siii(s_line), c='purple', linestyle='--')
                plt.plot(s_line / 10**4, chaos_nii_siii(s_line), c='lime', linestyle='--')
                plt.plot(s_line / 10**4, b24_nii_siii(s_line), c='fuchsia', linestyle='--')
                plt.plot(s_line  / 10**4, z21_nii_siii(s_line)  / 10**4, c='orangered', linestyle='--')
            
            if pq_list[i] == 'SIII_TEMP' and pq_list[j] == 'OII_TEMP':
                
                plt.plot(s_line  / 10**4, rv23_oii_siii(s_line), c='purple', linestyle='--')
                plt.plot(s_line  / 10**4, z21_oii_siii(s_line)  / 10**4, c='orangered', linestyle='--')
                
            if pq_list[i] == 'NII_TEMP' and pq_list[j] == 'OII_TEMP':
                
                plt.plot(s_line / 10**4, rv23_oii_nii(s_line), c='purple', linestyle='--')
                plt.plot(s_line  / 10**4, delgado_oii_nii(s_line)  / 10**4, c='cyan', linestyle='--')
                plt.plot(s_line  / 10**4, z21_oii_nii(s_line)  / 10**4, c='crimson', linestyle='--')

    if i != 0:
        l_bound = np.quantile(np.concatenate(lim_data) / 10**4, 0)
        h_bound = np.quantile(np.concatenate(lim_data) / 10**4, 1)
        up_val = 1.1
        down_val = 0.9
        axes[-1].set_ylim(l_bound*down_val, h_bound*up_val)
        axes[-1].set_xlim(l_bound*down_val, h_bound*up_val)

f_line = np.linspace(-10, -1, 10)
plt.plot(f_line, f_line, c='darkorange', linestyle='--', label='This Work')
plt.plot(f_line, f_line, c='crimson', linestyle='--', label='Zurita et al. 2021')
plt.plot(f_line, f_line, c='lime', linestyle='--', label='Rogers et al. 2022')
plt.plot(f_line, f_line, c='cyan', linestyle='--', label='Mendez-Delgado et al. 2023b')
plt.plot(f_line, f_line, c='purple', linestyle='--', label='Rickards Vaught et al. 2024')
plt.plot(f_line, f_line, c='fuchsia', linestyle='--', label='Brazzini et al. 2024')

plt.legend(bbox_to_anchor=(0.85, 1.05), fontsize = 11)

plt.subplots_adjust(wspace=0, hspace=0)
plt.show()

fig.savefig("/home/habjan/SITELLE/sitelle_metallicities/figures/temp_corner.png", bbox_inches="tight")

### Figure 3: density-density plot

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6,6), sharey=True)

oii_log = np.log10(strong_data['OII_DEN'])
oii_err = 1/np.log(10) * ((strong_data['OII_DEN_ERR']) / (strong_data['OII_DEN']))

sii_log = np.log10(strong_data['SII_DEN'])
sii_err = 1/np.log(10) * ((strong_data['SII_DEN_ERR']) / (strong_data['SII_DEN']))

for k in range(2, len(galdic)+1):

    plot_ind = np.where((strong_data['gal_name'] == galdic[k]) 
                        & (1 / (np.log(10) * oii_err) > snr) 
                        & (1 / (np.log(10) * sii_err) > snr)
                        & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]

    axs.errorbar(oii_log[plot_ind], sii_log[plot_ind],
                xerr = oii_err[plot_ind], yerr= sii_err[plot_ind], 
                fmt='none', ecolor='0.6', elinewidth=1, capsize=2, alpha=0.7, zorder=1)

    axs.scatter(oii_log[plot_ind], sii_log[plot_ind],
                s=mark_s, c=colordic[k], edgecolor='k', linewidth=0.4, zorder=2, marker='D')

plot_ind_1 = np.where((1 / (np.log(10) * oii_err) > snr) & (1 / (np.log(10) * sii_err) > snr))[0]

x, y = oii_log[plot_ind_1], sii_log[plot_ind_1]
x_err = oii_err[plot_ind_1]
y_err = sii_err[plot_ind_1]

axs.set_xlim(np.nanmin(np.concatenate([x, y]))*0.9, np.nanmax(np.concatenate([x, y]))*1.05)
axs.set_ylim(np.nanmin(np.concatenate([x, y]))*0.9, np.nanmax(np.concatenate([x, y]))*1.05)

spear, spear_err = spear_func(x, y, x_err, y_err)

fit_coef, fit_slope, fit_coef_err, fit_slope_err = WLS_func(x, y, x_err, y_err)

axs.plot(np.linspace(-1, 10**11, 10), fit_coef + fit_slope*np.linspace(-1, 10**11, 10), 
         c='darkorange', linestyle='--')
axs.fill_between(np.linspace(-1, 10**11, 10), 
                 (fit_coef - fit_coef_err) + (fit_slope - fit_slope_err)*np.linspace(-1, 10**11, 10), 
                 (fit_coef + fit_coef_err) + (fit_slope + fit_slope_err)*np.linspace(-1, 10**11, 10),
                 color='gold', alpha=0.3)

axs.text(1.07, 3.58, 
         r'$\rho$ = 'f'{np.round(spear, decimals=3)} 'r'$\pm$ 'f'{np.round(spear_err, decimals=3)}',
         fontsize = 12)

axs.plot(np.linspace(-2, 5, 100), np.linspace(-2, 5, 100), c='k')

axs.set_ylabel(r'$n_{e, [SII]}$ [$log(cm^{-3})$]', fontsize = 20)
axs.set_xlabel(r'$n_{e, [OII]7325+/3727+}$ [$log(cm^{-3})$]', fontsize = 20)

### Figure 4: O+ abundances

In [ ]:
seq = np.linspace(-3, 15, num = 20)
snr = 3
marker_oplus = 25
galdic = {1:'NGC4254', 6:'NGC3627', 5:'NGC4535', 4:'NGC3351', 3:'NGC2835', 2:'NGC0628'}
colordic = {2: 'red', 3:'k', 4:'blue', 5:'green', 6:'m'}

labels = np.array(['O2_NII_3727_SII', 'O2_NII_7325_SII', 'O2_OII_3727_SII', 'O2_NII_7325_OII'])

plot_type = 'paper'

if plot_type == 'paper':
    pq_labels = [
             r'$12 + log\left(O^{+}/H^{+}\right)$' + '\n' + r'$T_{e, [NII]},$ $n_{e, [OII]7325+/3727+},$ $[OII]\lambda3727$', 
             #r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [OII]7325+/3727+}$' + '\n' + r'$[OII]\lambda3727$',
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [SII]}$' + '\n' + r'$[OII]\lambda3727$',
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [SII]}$' + '\n' + r'$[OII]\lambda\lambda7325+$', 
             r'$T_{e, [OII]}$' + '\n' + r'$n_{e, [SII]}$' + '\n' + r'$[OII]\lambda3727$',
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [OII]7325+/3727+}$' + '\n' + r'$[OII]\lambda\lambda7325+$',]
    label_size = 12.5

elif plot_type == 'presentation':
    
    pq_labels = [
             r'$12 + log\left(O^{+}/H^{+}\right)$' + '\n' + r'$T_{e, [NII]},$' + '\n' + r'$n_{e, [OII]7325+/3727+},$' + '\n' + r'$[OII]\lambda3727$', 
             #r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [OII]7325+/3727+}$' + '\n' + r'$[OII]\lambda3727$',
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [SII]}$' + '\n' + r'$[OII]\lambda3727$',
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [SII]}$' + '\n' + r'$[OII]\lambda\lambda7325+$', 
             r'$T_{e, [OII]}$' + '\n' + r'$n_{e, [SII]}$' + '\n' + r'$[OII]\lambda3727$',
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [OII]7325+/3727+}$' + '\n' + r'$[OII]\lambda\lambda7325+$',]
    label_size = 20

plt.figure(figsize=(15, 20))
axes = []

for j in range(len(labels)):
    if j == 0 :
        axes.append(plt.subplot(5,4,j+1))
    else:
        axes.append(plt.subplot(5,4,j+1, sharex=axes[0], sharey=axes[0]))
    
    if j%4 == 1 or j%4 == 2 or j%4 == 3:
        axes[j].tick_params('y', labelleft=False)
    
    axes[-1].set_xlabel(r'$12 + log\left(O^{+}/H^{+}\right)$', fontsize = label_size)
    axes[-1].text(0.95, 0.05, pq_labels[j+1], color='k', ha="right", transform=axes[j].transAxes, 
                  fontweight = 'bold', fontsize = label_size)
    if j == 0:
        axes[-1].set_ylabel(pq_labels[0], fontsize= label_size)
    
    axes[-1].plot(np.linspace(6, 10, 10), np.linspace(6, 10, 10), c='k', alpha=1)

    lim_list = []

    for i in range(2, len(galdic) + 1):
        plot_bool = np.where((strong_data['gal_name'] == galdic[i]) 
                             & (strong_data['O2_NII_3727_OII']/strong_data['O2_NII_3727_OII_ERR'] > snr) 
                             & (strong_data[labels[j]]/strong_data[labels[j]+'_ERR'] > snr)
                             & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]

        axes[j].errorbar(12+np.log10(strong_data[labels[j]][plot_bool]), 12+np.log10(strong_data['O2_NII_3727_OII'][plot_bool]),
                        xerr=1/np.log(10) * (strong_data[labels[j]+'_ERR'][plot_bool] / strong_data[labels[j]][plot_bool]),
                        yerr= 1/np.log(10) * (strong_data['O2_NII_3727_OII_ERR'][plot_bool] / strong_data['O2_NII_3727_OII'][plot_bool]),
                        fmt='none', ecolor='0.6', elinewidth=1, capsize=2, alpha=0.7, zorder=1)

        axes[j].scatter(12+np.log10(strong_data[labels[j]][plot_bool]), 12+np.log10(strong_data['O2_NII_3727_OII'][plot_bool]),
                        s=marker_oplus, c=colordic[i], edgecolor='k', linewidth=0.4, zorder=2, marker='D')
        
        lim_list.append(np.array(12+np.log10(strong_data[labels[j]][plot_bool]))) 
        lim_list.append(np.array(12+np.log10(strong_data['O2_NII_3727_OII'][plot_bool])))  

    lim_list = np.concatenate(lim_list)

    axes[-1].set_ylim(np.nanquantile(lim_list, 0)*0.88, np.nanquantile(lim_list, 1)*1.05)
    axes[-1].set_xlim(np.nanquantile(lim_list, 0)*0.88, np.nanquantile(lim_list, 1)*1.05)

    #axes[-1].text(0.475, 10, pq_labels[j+1])

    #axes[j].legend(fontsize=9)

plt.subplots_adjust(wspace=0.05,hspace=0)

### Figure 5: O2+ abundances

In [ ]:
seq = np.linspace(-3, 15, num = 20)
snr = 3
galdic = {1:'NGC4254', 6:'NGC3627', 5:'NGC4535', 4:'NGC3351', 3:'NGC2835', 2:'NGC0628'}
colordic = {2: 'red', 3:'k', 4:'blue', 5:'green', 6:'m'}

labels = np.array(['O3_T0_SII', 'O3_OIII_SII', 'O3_OIII_OII'])

plot_type = 'paper'

if plot_type == 'paper':
    
    pq_labels = [
        r'$12 + log\left(O^{2+}/H^{+}\right)$' + '\n' + r'$T_{0}(O^{2+}),$ $n_{e, [OII]7325+/3727+}$', 
        r'$T_{0}(O^{2+})$' + '\n' + r'$n_{e, [SII]}$',
        r'$T_{e, [OIII]}$' + '\n' + r'$n_{e, [SII]}$',
        r'$T_{e, [OIII]}$' + '\n' + r'$n_{e, [OII]7325+/3727+}$']
    label_size = 12.5

elif plot_type == 'presentation':
    
    pq_labels = [
        r'$12 + log\left(O^{2+}/H^{+}\right)$' + '\n' + r'$T_{0}(O^{2+})$' + '\n' + r'$n_{e, [OII]7325+/3727+}$', 
        r'$T_{0}(O^{2+}),$' + '\n' + r'$n_{e, [SII]}$',
        r'$T_{e, [OIII]},$' + '\n' + r'$n_{e, [SII]}$',
        r'$T_{e, [OIII]},$' + '\n' + r'$n_{e, [OII]7325+/3727+}$']
    label_size = 20

plt.figure(figsize=(15, 20))
axes = []

for j in range(len(labels)):
    if j == 0 :
        axes.append(plt.subplot(5,4,j+1))
    else:
        axes.append(plt.subplot(5,4,j+1, sharex=axes[0], sharey=axes[0]))
    
    if j%4 == 1 or j%4 == 2 or j%4 == 3:
        axes[j].tick_params('y', labelleft=False)
    
    axes[-1].set_xlabel(r'$12 + log\left(O^{2+}/H^{+}\right)$', fontsize = label_size)
    axes[-1].text(0.95, 0.05, pq_labels[j+1], color='k', ha="right", transform=axes[j].transAxes, 
                  fontweight = 'bold', fontsize = label_size)
    if j == 0:
        axes[-1].set_ylabel(pq_labels[0], fontsize= label_size)
    
    axes[-1].plot(np.linspace(6, 10, 10), np.linspace(6, 10, 10), c='k', alpha=1)

    lim_list = []

    for i in range(2, len(galdic) + 1):
        plot_bool = np.where((strong_data['gal_name'] == galdic[i]) 
                             & (strong_data['O3_T0_OII']/strong_data['O3_T0_OII_ERR'] > snr) 
                             & (strong_data[labels[j]]/strong_data[labels[j]+'_ERR'] > snr)
                             & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]

        axes[j].errorbar(12+np.log10(strong_data[labels[j]][plot_bool]), 12+np.log10(strong_data['O3_T0_OII'][plot_bool]),
                         xerr=1/np.log(10) * (strong_data[labels[j]+'_ERR'][plot_bool] / strong_data[labels[j]][plot_bool]),
                         yerr= 1/np.log(10) * (strong_data['O3_T0_OII_ERR'][plot_bool] / strong_data['O3_T0_OII'][plot_bool]), 
                        fmt='none', ecolor='0.6', elinewidth=1, capsize=2, alpha=0.7, zorder=1)

        axes[j].scatter(12+np.log10(strong_data[labels[j]][plot_bool]), 12+np.log10(strong_data['O3_T0_OII'][plot_bool]),
                        s=mark_s, c=colordic[i], edgecolor='k', linewidth=0.4, zorder=2, marker='D')
        
        lim_list.append(np.array(12+np.log10(strong_data[labels[j]][plot_bool]))) 
        lim_list.append(np.array(12+np.log10(strong_data['O3_T0_OII'][plot_bool])))  

    lim_list = np.concatenate(lim_list)

    axes[-1].set_ylim(np.nanquantile(lim_list, 0)*0.94, np.nanquantile(lim_list, 1)*1.02)
    axes[-1].set_xlim(np.nanquantile(lim_list, 0)*0.94, np.nanquantile(lim_list, 1)*1.02)

plt.subplots_adjust(wspace=0.05,hspace=0)

### Figure 6: total metallicities

In [ ]:
seq = np.linspace(-3, 15, num = 20)
snr = 3
marker_met = 25
galdic = {1:'NGC4254', 6:'NGC3627', 5:'NGC4535', 4:'NGC3351', 3:'NGC2835', 2:'NGC0628'}
colordic = {2: 'red', 3:'k', 4:'blue', 5:'green', 6:'m'}

labels = np.array(['OH_T0_OII_NII_OII_3727', 'OH_T0_SII_NII_SII_3727', 'OH_OIII_SII_NII_SII_7325', 
                   'OH_OIII_OII_NII_OII_3727' ])

pq_labels = [r'$12+log(O/H)$' + '\n' + r'$MD23$', 
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [OII]7325+/3727+}$' + '\n' + r'$[OII]\lambda3727$' + '\n' + '\n' + r'$T_{0}(O^{2+})$' + '\n' + r'$n_{e, [OII]7325+/3727+}$',
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [SII]}$' + '\n' + r'$[OII]\lambda3727$' + '\n' + '\n' + r'$T_{0}(O^{2+})$' + '\n' + r'$n_{e, [SII]}$',
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [SII]}$' + '\n' + r'$[OII]\lambda\lambda7320$' + '\n' + '\n' + r'$T_{e, [OIII]}$' + '\n' + r'$n_{e, [SII]}$', 
             r'$T_{e, [NII]}$' + '\n' + r'$n_{e, [OII]7325+/3727+}$' + '\n' + r'$[OII]\lambda3727$' + '\n' + '\n' + r'$T_{e, [OIII]}$' + '\n' + r'$n_{e, [OII]7325+/3727+}$',
             ]

label_size = 12.5

plt.figure(figsize=(18.5, 20))
axes = []
lim_list = []

for j in range(len(labels)):
    if j == 0 :
        axes.append(plt.subplot(5,5,j+1))
    else:
        axes.append(plt.subplot(5,5,j+1, sharex=axes[0], sharey=axes[0]))
    
    if j%5 == 1 or j%5 == 2 or j%5 == 3 or j%5 == 4:
        axes[j].tick_params('y', labelleft=False)
    
    axes[-1].set_xlabel(r'$12+log(O/H)$', fontsize = label_size)
    axes[-1].text(0.95, 0.05, pq_labels[j+1], color='k', ha="right", transform=axes[j].transAxes, 
                  fontweight = 'bold', fontsize = label_size)
    if j == 0:
        axes[-1].set_ylabel(pq_labels[0], fontsize= label_size)
    
    axes[-1].plot(np.linspace(-10, 20, 10), np.linspace(-10, 20, 10), c='k', alpha=1)

    for i in range(2, len(galdic) + 1):
        plot_bool = np.where((strong_data['gal_name'] == galdic[i]) 
                             & (strong_data[labels[j]]/strong_data[labels[j]+'_ERR'] > snr)
                             & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]
        
        axes[j].errorbar(strong_data[labels[j]][plot_bool], strong_data['met_md23'][plot_bool],
                         xerr=strong_data[labels[j]+'_ERR'][plot_bool],
                         yerr= strong_data['met_md23_err'][plot_bool],  
                        fmt='none', ecolor='0.6', elinewidth=1, capsize=2, alpha=0.7, zorder=1)

        axes[j].scatter(strong_data[labels[j]][plot_bool], strong_data['met_md23'][plot_bool],
                        s=marker_met, c=colordic[i], edgecolor='k', linewidth=0.4, zorder=2, marker='D')
        
        
        lim_list.append(np.array(strong_data[labels[j]][plot_bool])) 
        lim_list.append(np.array(strong_data['met_md23'][plot_bool]))  

lim_list_1 = np.concatenate(lim_list)

axes[-1].set_ylim(np.nanmin(lim_list_1)*0.99, np.nanmax(lim_list_1)*1.01)
axes[-1].set_xlim(np.nanmin(lim_list_1)*0.99, np.nanmax(lim_list_1)*1.01)

plt.subplots_adjust(wspace=0.05,hspace=0)

### Figure 7: N/O comparisons

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharey=True, sharex=True)

for k in range(2, len(galdic)+1):

    plot_ind = np.where((strong_data['gal_name'] == galdic[k]) 
                        & (strong_data['N2_ABUN_NII_SII']/strong_data['N2_ABUN_NII_SII_ERR'] > snr) 
                        & (strong_data['N2_ABUN_NII_OII']/strong_data['N2_ABUN_NII_OII_ERR'] > snr)
                        & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]
    
    axs[0].errorbar(np.array(np.log10(strong_data['N2_ABUN_NII_SII'][plot_ind])), np.array(np.log10(strong_data['N2_ABUN_NII_OII'][plot_ind])),
                    xerr=1/np.log(10) * (strong_data['N2_ABUN_NII_SII_ERR'][plot_ind] / strong_data['N2_ABUN_NII_SII'][plot_ind]),
                    yerr=1/np.log(10) * (strong_data['N2_ABUN_NII_OII_ERR'][plot_ind] / strong_data['N2_ABUN_NII_OII'][plot_ind]),   
                    fmt='none', ecolor='0.6', elinewidth=1, capsize=2, alpha=0.7, zorder=1)

    axs[0].scatter(np.log10(strong_data['N2_ABUN_NII_SII'][plot_ind]), np.log10(strong_data['N2_ABUN_NII_OII'][plot_ind]),
                    s=mark_s, c=colordic[k], edgecolor='k', linewidth=0.4, zorder=2, marker='D')
    

plot_ind_1 = np.where((strong_data['N2_ABUN_NII_SII']/strong_data['N2_ABUN_NII_SII_ERR'] > snr) 
                      & (strong_data['N2_ABUN_NII_OII']/strong_data['N2_ABUN_NII_OII_ERR'] > snr)
                      & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]
lim_list = np.concatenate([np.log10(np.array(strong_data['N2_ABUN_NII_SII'][plot_ind_1])), np.log10(np.array(strong_data['N2_ABUN_NII_OII'][plot_ind_1]))])

axs[0].plot(np.linspace(-2, 1, 100), np.linspace(-2, 1, 100), c='k')
axs[0].set_ylabel(r'$log(N^{+}/O^{+})$' + '\n' + r'$T_{e, [NII]}, n_{e, [OII]7325+/3727+}$', fontsize=15)
axs[0].set_xlabel(r'$log(N^{+}/O^{+})$' + '\n' + r'$T_{e, [NII]}, n_{e, [SII]}$', fontsize=15)

x, y = np.log10(np.array(strong_data['N2_ABUN_NII_SII'][plot_ind_1])), np.log10(np.array(strong_data['N2_ABUN_NII_OII'][plot_ind_1]))
x_err = 1/np.log(10) * (strong_data['N2_ABUN_NII_SII_ERR'][plot_ind_1] / strong_data['N2_ABUN_NII_SII'][plot_ind_1])
y_err = 1/np.log(10) * (strong_data['N2_ABUN_NII_OII_ERR'][plot_ind_1] / strong_data['N2_ABUN_NII_OII'][plot_ind_1])
spear, spear_err = spear_func(x, y, x_err, y_err)
fit_coef, fit_slope, fit_coef_err, fit_slope_err = WLS_func(x, y, x_err, y_err)

axs[0].plot(np.linspace(-5, 5, 10), fit_coef + fit_slope*np.linspace(-5, 5, 10), 
             c='darkorange', linestyle='--') 

axs[0].fill_between(np.linspace(-5, 5, 10), 
                     (fit_coef - fit_coef_err) + (fit_slope - fit_slope_err)*np.linspace(-5, 5, 10), 
                     (fit_coef + fit_coef_err) + (fit_slope + fit_slope_err)*np.linspace(-5, 5, 10),
                     color='gold', alpha=0.3)
axs[0].text(-1.1, -0.43, r'$\rho$ = 'f'{np.round(spear, decimals=3)} 'r'$\pm$ 'f'{np.round(spear_err, decimals=3)}', fontsize = 15)


plot_ind_3 = np.where((strong_data['N2_ABUN_NII_OII']/strong_data['N2_ABUN_NII_OII_ERR'] > snr) 
                      & (1 / (np.log(10) * strong_data['N_NII_3727_OII_T0_OII_ERR']) > snr)
                      & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]
lim_list = np.concatenate([lim_list, np.log10(np.array(strong_data['N2_ABUN_NII_OII'][plot_ind_3])), np.array(strong_data['N_NII_3727_OII_T0_OII'][plot_ind_3])])

for k in range(2, len(galdic)+1):

    plot_ind = np.where((strong_data['gal_name'] == galdic[k]) 
                        & (strong_data['N2_ABUN_NII_OII']/strong_data['N2_ABUN_NII_OII_ERR'] > snr) 
                        & (1 / (np.log(10) * strong_data['N_NII_3727_OII_T0_OII_ERR']) > snr)
                        & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]

    axs[1].errorbar(strong_data['N_NII_3727_OII_T0_OII'][plot_ind], np.log10(strong_data['N2_ABUN_NII_OII'][plot_ind]),
             yerr=1/np.log(10) * (strong_data['N2_ABUN_NII_OII_ERR'][plot_ind] / strong_data['N2_ABUN_NII_OII'][plot_ind]),
             xerr=strong_data['N_NII_3727_OII_T0_OII_ERR'][plot_ind],  
                    fmt='none', ecolor='0.6', elinewidth=1, capsize=2, alpha=0.7, zorder=1)

    axs[1].scatter(strong_data['N_NII_3727_OII_T0_OII'][plot_ind], np.log10(strong_data['N2_ABUN_NII_OII'][plot_ind]),
                    s=mark_s, c=colordic[k], edgecolor='k', linewidth=0.4, zorder=2, marker='D')

axs[1].plot(np.linspace(-2, 1, 100), np.linspace(-2, 1, 100), c='k')
axs[1].set_xlabel(r'$log(N/O)$' + '\n' + r'$T_{e, [NII]}, T_{0}(O^{2+}), n_{e, [OII]7325+/3727+}$', fontsize=15)

axs[1].set_xlim(np.nanmin(lim_list)*1.15, np.nanmax(lim_list)*0.85)
axs[1].set_ylim(np.nanmin(lim_list)*1.15, np.nanmax(lim_list)*0.85)

x, y = np.array(strong_data['N_NII_3727_OII_T0_OII'][plot_ind_3]), np.log10(np.array(strong_data['N2_ABUN_NII_OII'][plot_ind_3]))
x_err = np.array(strong_data['N_NII_3727_OII_T0_OII_ERR'][plot_ind_3])
y_err = 1/np.log(10) * (strong_data['N2_ABUN_NII_OII_ERR'][plot_ind_3] / strong_data['N2_ABUN_NII_OII'][plot_ind_3])
spear, spear_err = spear_func(x, y, x_err, y_err)
fit_coef, fit_slope, fit_coef_err, fit_slope_err = WLS_func(x, y, x_err, y_err)

axs[1].plot(np.linspace(-5, 5, 10), fit_coef + fit_slope*np.linspace(-5, 5, 10), 
             c='darkorange', linestyle='--')

axs[1].fill_between(np.linspace(-5, 5, 10), 
                     (fit_coef - fit_coef_err) + (fit_slope - fit_slope_err)*np.linspace(-5, 5, 10), 
                     (fit_coef + fit_coef_err) + (fit_slope + fit_slope_err)*np.linspace(-5, 5, 10),
                     color='gold', alpha=0.3)
axs[1].text(-1.1, -0.43, r'$\rho$ = 'f'{np.round(spear, decimals=3)} 'r'$\pm$ 'f'{np.round(spear_err, decimals=3)}', fontsize = 15)

plt.subplots_adjust(wspace=0.05, hspace=0)
plt.show()

### Figure 8: N/O verus O/H

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

N_abun = 'N2_ABUN_NII_SII'

met, met_err = 'OH_T0_OII_NII_OII_3727', 'OH_T0_OII_NII_OII_3727_ERR'

def nicholls_rel(line_in):
    return np.log10(10**-1.732 + 10**(line_in + 2.19 - 12))

def morisset_rel(line_in):
    return -16.09 + 1.81 * (line_in)

def pilyugin_rel(line_in):
    
    z0 = 8.23 
    
    if line_in < z0:
        val = -1.39 + 0.39(line_in-z0) + 0.30*(line_in-z0)**2
        return val
    
    elif line_in > z0:
        val -1.39 + 1.24(line_in-z0) + 1.63*(line_in-z0)**2
        return val

n_line = np.linspace(6, 9, 20)

axs[0].plot(n_line, nicholls_rel(n_line), c='darkorange', label='Nicholls et al. 2016',
            linewidth=3)

n_line = np.linspace(8.2, 8.7, 20)

axs[0].plot(n_line, np.array([morisset_rel(n_line[i]) for i in range(len(n_line))]), c='skyblue', label='Morisset et al. 2016',
            linewidth=3)

for k in range(2, len(galdic)+1):

    plot_ind = np.where((strong_data['gal_name'] == galdic[k]) 
                        & (strong_data[N_abun] / strong_data[N_abun+'_ERR'] > snr)
                        & (1 / (np.log(10) * strong_data[met_err]) > snr)
                        & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]
    
    axs[0].errorbar(strong_data[met][plot_ind], np.log10(strong_data[N_abun][plot_ind]),
                 yerr= strong_data[N_abun+'_ERR'][plot_ind] / (np.log(10) * strong_data[N_abun][plot_ind]),
                 xerr=strong_data[met_err][plot_ind],
                    fmt='none', ecolor='0.6', elinewidth=1, capsize=2, alpha=0.7, zorder=1)

    axs[0].scatter(strong_data[met][plot_ind], np.log10(strong_data[N_abun][plot_ind]),
                    s=mark_s, c=colordic[k], edgecolor='k', linewidth=0.4, zorder=2, marker='D')

plot_ind_2 = np.where((strong_data[N_abun] / strong_data[N_abun+'_ERR'] > snr)
                      & (1 / (np.log(10) * strong_data[met_err]) > snr)
                      & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]

lim_list = np.array(strong_data[met][plot_ind_2])
axs[0].set_xlim(8, 9.15)

n_line = np.linspace(7.1, 8.8, 20)


x, y = np.array(strong_data[met][plot_ind_2]), np.array(strong_data[N_abun][plot_ind_2])
x_err = np.array(strong_data[met_err][plot_ind_2])
y_err = np.array(strong_data[N_abun+'_ERR'][plot_ind_2]) / (np.log(10) * np.array(strong_data[N_abun][plot_ind_2]))
spear, spear_err = spear_func(x, y, x_err, y_err)
fit_coef, fit_slope, fit_coef_err, fit_slope_err = WLS_func(x, y, x_err, y_err)

axs[0].text(0.02, 0.98, rf'$\rho$ = {spear:.3f} $\pm$ {spear_err:.3f}', fontsize = 15, transform=axs[0].transAxes, ha='left', va='top')

axs[0].set_ylabel(r'log$(N^{+}/O^{+})$' + '\n' + r'$T_{e, [NII]}, n_{e, [SII]}$', fontsize = 15)
axs[0].set_xlabel(r'12 + log(O/H) ($n_{e, [SII]}$)', fontsize = 15)
axs[0].legend(loc='lower right')

N_abun = 'N2_ABUN_NII_OII'
met, met_err = 'OH_T0_SII_NII_SII_3727', 'OH_T0_SII_NII_SII_3727_ERR'

n_line = np.linspace(6, 9, 20)

axs[1].plot(n_line, nicholls_rel(n_line), c='darkorange', label='Nicholls et al. 2016',
            linewidth=3)

n_line = np.linspace(8.2, 8.7, 20)

axs[1].plot(n_line, np.array([morisset_rel(n_line[i]) for i in range(len(n_line))]), c='skyblue', label='Morisset et al. 2016',
            linewidth=3)

for k in range(2, len(galdic)+1):

    plot_ind = np.where((strong_data['gal_name'] == galdic[k]) 
                        & (strong_data[N_abun] / strong_data[N_abun+'_ERR'] > snr)
                        & (1 / (np.log(10) * strong_data[met_err]) > snr)
                        & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]
    
    axs[1].errorbar(strong_data[met][plot_ind], np.log10(strong_data[N_abun][plot_ind]),
                 yerr= strong_data[N_abun+'_ERR'][plot_ind],
                 xerr=strong_data[met_err][plot_ind],
                    fmt='none', ecolor='0.6', elinewidth=1, capsize=2, alpha=0.7, zorder=1)

    axs[1].scatter(strong_data[met][plot_ind], np.log10(strong_data[N_abun][plot_ind]),
                    s=mark_s, c=colordic[k], edgecolor='k', linewidth=0.4, zorder=2, marker='D')

plot_ind_2 = np.where((strong_data[N_abun] / strong_data[N_abun+'_ERR'] > snr) 
                      & (1 / (np.log(10) * strong_data[met_err]) > snr)
                      & (np.array(strong_data['OII3727_FLUX_CORR']) > 10**6))[0]

lim_list = np.array(strong_data[met][plot_ind_2])
axs[1].set_xlim(8, 9.15)
axs[1].set_ylim(-1.5, -0.25)

n_line = np.linspace(7.1, 8.8, 20)

x, y = np.array(strong_data[met][plot_ind_2]), np.array(strong_data[N_abun][plot_ind_2])
x_err = np.array(strong_data[met_err][plot_ind_2])
y_err = np.array(strong_data[N_abun+'_ERR'][plot_ind_2])
spear, spear_err = spear_func(x, y, x_err, y_err)
fit_coef, fit_slope, fit_coef_err, fit_slope_err = WLS_func(x, y, x_err, y_err)

axs[1].text(0.02, 0.98, rf'$\rho$ = {spear:.3f} $\pm$ {spear_err:.3f}', fontsize = 15, transform=axs[1].transAxes, ha='left', va='top')

axs[1].set_ylabel(r'$log(N^{+}/O^{+})$' + '\n' + r'$T_{e, [NII]}, n_{e, [OII]7325+/3727+}$', fontsize = 15)
axs[1].set_xlabel(r'12 + log(O/H) ($n_{e, [OII]7325+/3727+}$)', fontsize = 15)
axs[1].legend(loc='lower right')

plt.subplots_adjust(wspace=0.2, hspace=0)
plt.show()

### Figure 9: ADF proxy versus density, colored by metallicity

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))

snr = 2
label_size = 25
tick_size = 14
cbar_size = 20
mark_s = 50

# ADF ratio and uncertainty
adf = np.array(strong_data['O3_T0_OII'] / strong_data['O3_OIII_OII'])
adf_err = adf * np.sqrt(
    (strong_data['O3_T0_OII_ERR'] / strong_data['O3_T0_OII'])**2 +
    (strong_data['O3_OIII_OII_ERR'] / strong_data['O3_OIII_OII'])**2
)

# log10 SII density and uncertainty in log10 space
den = np.array(np.log10(strong_data['SII_DEN']))
den_err = np.array((1 / np.log(10)) * (strong_data['SII_DEN_ERR'] / strong_data['SII_DEN']))

# Color by Metallicity
oiii_oii = np.array((strong_data['met_md23']))

# Selection
plot_ind = np.where(
    (adf / adf_err > snr) &
    (1 / (np.log(10) * den_err) > snr) &
    np.isfinite(adf) & np.isfinite(adf_err) &
    np.isfinite(den) & np.isfinite(den_err) &
    np.isfinite(oiii_oii) & (oiii_oii > 0)
)[0]

x = den[plot_ind]
y = adf[plot_ind]
x_err = den_err[plot_ind]
y_err = adf_err[plot_ind]
c = oiii_oii[plot_ind]

# Draw error bars first
ax.errorbar(
    x, y,
    xerr=x_err, yerr=y_err,
    fmt='none',
    ecolor='0.6',
    elinewidth=1,
    capsize=2,
    alpha=0.7,
    zorder=1
)

# Scatter on top, colored by [OIII]/[OII]
sc = ax.scatter(
    x, y,
    c=c,
    s=mark_s,
    cmap='viridis',
    edgecolor='k',
    linewidth=0.4,
    zorder=2,
    marker='D'
)

# Correlation on plotted sample
spear, spear_err = spear_func(x, y, x_err, y_err)
label = rf'$\rho$ = {spear:.3f} $\pm$ {spear_err:.3f}'
ax.text(0.025, 0.96, label, fontsize=14, transform=ax.transAxes, va='top')

# Labels and limits
ax.set_xlabel(r"$n_{e, [S\,II]}$ $[log(cm^{-3})]$", fontsize=label_size)
ax.set_ylabel(
    r'$\frac{O^{2+}\left(T_{0}(O^{2+}),\,n_{e,[O\,II]7325+/3727+}\right)}'
    r'{O^{2+}\left(T_{e,[O\,III]},\,n_{e,[O\,II]7325+/3727+}\right)}$',
    fontsize=label_size
)

ax.set_xlim(np.nanmin(x) * 0.9, np.nanmax(x) * 1.1)
ax.set_ylim(np.nanmin(y) * 0.5, np.nanmax(y) * 1.25)

ax.tick_params(axis='both', labelsize=tick_size)

cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label(r'MD23 $[12 + log(O/H)]$', fontsize=cbar_size)
cbar.ax.tick_params(labelsize=13)

plt.tight_layout()
plt.show()